## System Prompt Best Practices

The system prompt is the single highest-leverage piece of text in any LLM application — it sets persistent behaviour for every single call. Writing it well is a distinct, learnable craft, separate from context engineering (which is about WHAT information surrounds the prompt, not HOW the instructions themselves are written).

## How it actually works

**Why the system prompt deserves more deliberate care than a quick instruction:**
Because it persists across every single call using that prompt, small wording
improvements compound — a slightly better system prompt improves EVERY output
generated under it, not just one. This is exactly why your Personal Brand Autopilot's
prompts (DRAFTER_PROMPT, VOICE_GUARDIAN_PROMPT, etc.) went through multiple rounds
of refinement rather than being written once and left alone — each refinement round
improved every future post generated under that prompt, retroactively justifying
the time spent iterating.

**Be specific and concrete, not vague — the single highest-impact practice:**
"Write professionally" is vague and gives the model little to actually act on.
"Use short sentences, no corporate buzzwords, never say 'I'm excited to share'"
is specific and gives the model concrete, checkable behaviour to follow. Your own
VOICE_GUARDIAN_PROMPT demonstrates this well — it does not just say "sound authentic,"
it lists exact banned phrases and exact tone descriptors.

**Show, don't just tell — examples often outperform descriptions:**
Describing a desired tone in the abstract is weaker than showing one or two concrete
examples of exactly the kind of output you want. This is why your DRAFTER_PROMPT
includes the "gold standard" example lines — those examples likely do more work
than the surrounding descriptive instructions alone would.

**Separate the WHO from the WHAT from the HOW:**
A well-structured system prompt typically separates: who the model is acting as
(persona/role), what it should accomplish (the task), and how it should accomplish
it (constraints, format, tone). Mixing these together in unstructured prose makes
it harder for the model to weigh each instruction correctly, and harder for YOU
to debug which part of the prompt is producing a specific behaviour.

**Put the most important constraints near the beginning AND reinforce critical ones near the end:**
This connects directly to "Lost in the Middle" (covered later in this folder) —
a system prompt that buries its most critical constraint in the middle of a long
block of less important instructions risks that constraint getting less attention
than instructions positioned at the start or end.

## Where it breaks or gets dangerous

**Overloading one system prompt with too many competing jobs:**
If a single system prompt is trying to define persona, enforce a strict format,
apply a writing voice, AND filter for compliance issues all at once, quality on
each individual job tends to degrade — this is conceptually similar to why your
own multi-agent pipeline exists at all. Splitting jobs across the Drafter, Strategist,
and Voice Guardian rather than cramming everything into one giant prompt is a
genuine application of this principle, even if it was not framed this way originally.

**Vague negative instructions are weaker than specific ones:**
"Don't sound like an AI" is vague and hard for the model to act on consistently.
"Never use the phrases 'I'm excited to share' or 'game changer'; never use the
word 'leverage'" is specific and directly actionable. Your VOICE_GUARDIAN_PROMPT's
explicit banned-phrase list demonstrates exactly the right level of specificity here.

**System prompts can drift out of sync with how the application actually evolved:**
If you change what an agent is supposed to do but don't update the system prompt
to match, you get a persistent mismatch between the prompt's instructions and the
agent's actual intended behaviour — a subtle bug that does not throw an error,
it just silently produces slightly-wrong output indefinitely until someone notices.

**Treating the system prompt as a complete security boundary — a real risk to flag:**
This connects directly to the previous notebook's point: a system prompt heavily
increases the LIKELIHOOD of compliant behaviour, but it is not an absolute guarantee,
especially against deliberate prompt injection. For genuinely high-stakes constraints
(legal, financial, safety), the system prompt should be paired with independent
output validation/guardrails, not relied on alone.

**Length tradeoffs are real — longer is not automatically better:**
An extremely long, exhaustive system prompt consumes context window budget on
every single call, and past a certain point, additional instructions can start
competing with each other for the model's attention rather than each adding
clean, independent value.

In [ ]:
# System Prompt Best Practices — illustrating vague vs specific instructions directly

import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

task_prompt = "Write a two-sentence LinkedIn post about learning machine learning while working in D365."

vague_system_prompt = "Write professionally and sound authentic, not like an AI."

specific_system_prompt = """Write in this voice:
- Short sentences, direct, no fluff
- Never say "I'm excited to share", "game changer", "thrilled", or "passionate"
- Never use the word "leverage"
- Sound like a senior consultant, not a student describing what they learned
- One specific, concrete detail beats a general statement every time"""

print("=== VAGUE system prompt ===")
print(f"System: {vague_system_prompt}")
response_vague = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=100,
    system=vague_system_prompt,
    messages=[{"role": "user", "content": task_prompt}]
)
print("Output:", response_vague.content[0].text)
print()

print("=== SPECIFIC system prompt ===")
print(f"System: {specific_system_prompt}")
response_specific = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=100,
    system=specific_system_prompt,
    messages=[{"role": "user", "content": task_prompt}]
)
print("Output:", response_specific.content[0].text)
print()

print("Compare the two outputs above — the specific prompt typically produces")
print("output with fewer generic AI-sounding phrases and a more distinctive voice,")
print("simply because it gave the model concrete, checkable behaviour to follow")
print("rather than an abstract quality ('professional', 'authentic') to interpret.")

## My D365 analogy

Writing a good system prompt is similar to writing a good approval workflow
configuration versus a vague business requirement document.

"Make sure invoices get approved appropriately" is the equivalent of a vague system
prompt — technically a statement of intent, but it gives the system nothing concrete
to actually act on. Every implementation of "appropriately" will differ, and debugging
why behaviour is inconsistent becomes genuinely difficult, because the instruction
itself was never specific enough to be wrong or right against.

"Route to finance manager if amount exceeds $50,000 OR if vendor is flagged as
high-risk; otherwise auto-approve if three-way match passes within 2% tolerance"
is the equivalent of a specific, well-written system prompt — concrete, checkable,
and debuggable. If something goes wrong, you can trace exactly which condition
fired and why.

The skill of writing a precise functional specification for a workflow configuration
and the skill of writing a precise system prompt for an LLM are, underneath the
different vocabulary, almost the same skill: translate a vague intention into
specific, checkable, unambiguous instructions.b

## LinkedIn post idea

"Sound professional and authentic" is a terrible instruction to give an AI model.
Not because the goal is wrong — because it gives the model nothing concrete to
actually act on.

"Never say 'I'm excited to share'. Never use the word 'leverage'. Short sentences,
no fluff." is a genuinely better instruction for the exact same underlying goal —
specific, checkable, and the model can actually follow it consistently.

This is not really an AI-specific insight. It is the same lesson every consultant
learns writing functional requirements. "Make sure approvals work appropriately"
produces inconsistent results because nobody, human or AI, can act on "appropriately"
in a checkable way. "Route to finance manager if amount exceeds threshold X"
produces consistent results because there is something concrete to check against.

I spent years learning to write specific functional specs instead of vague business
requirements. Turns out writing a good system prompt for an AI model is the exact
same discipline wearing different vocabulary.